# 1. Trajectory

Generate and visualize the paired-spoke spiral phyllotaxis trajectory used by ZTE-STAR.


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add the repository root to the import path when the notebook is launched
# directly from the repository rather than from an installed package.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import config, trajectory

cfg = config.get()
print(
    f"ZTE-STAR: M={cfg.M}, P={cfg.P}, k={cfg.k}, "
    f"total spokes={cfg.n_spokes}"
)
print(
    f"Conventional radial: {cfg.radial_n_pro} spokes per repetition"
)


In [ ]:
def spoke_directions(traj):
    """Extract one unit direction vector for each spoke.

    Args:
        traj: Trajectory array with shape `(3, n_readout, n_spokes)`.

    Returns:
        Array with shape `(3, n_spokes)` containing normalized endpoint
        directions.
    """
    endpoints = traj[:, -1, :]
    return endpoints / np.linalg.norm(endpoints, axis=0)


def lon_lat(directions):
    """Convert Cartesian unit vectors to longitude and latitude.

    Args:
        directions: Array with shape `(3, n_spokes)`.

    Returns:
        A tuple `(longitude, latitude)` in radians, suitable for a
        Mollweide projection.
    """
    longitude = np.arctan2(directions[1], directions[0])
    latitude = np.arcsin(np.clip(directions[2], -1, 1))
    return longitude, latitude


# Store only the paired-spoke trajectories required by the reconstruction
# notebook. Acquisition-order trajectories are kept in memory for plotting.
out = config.OUTPUT_DIR / "trajectory"
out.mkdir(parents=True, exist_ok=True)

start_time = time.time()
spiral = trajectory.from_settings(
    cfg,
    "spiral",
    pairspoke=False,
    dtype=np.float32,
)
spiral_pair = trajectory.from_settings(
    cfg,
    "spiral",
    pairspoke=True,
    dtype=np.float32,
)
radial = trajectory.from_settings(
    cfg,
    "radial",
    pairspoke=False,
    dtype=np.float32,
)
radial_pair = trajectory.from_settings(
    cfg,
    "radial",
    pairspoke=True,
    dtype=np.float32,
)

np.save(out / "spiral_phyllotaxis_pairspoke.npy", spiral_pair)
np.save(out / "bruker_radial_pairspoke.npy", radial_pair)
print(f"Trajectories generated in {time.time() - start_time:.1f} s")


## Paired spokes

Each positive spoke is paired with its opposite direction for algebraic dead-time infilling.


In [ ]:
# Select one positive spoke and its corresponding negative partner within
# the first interleaf. The two directions should be antiparallel.
positive_index = 30
negative_index = positive_index + cfg.P
directions = spoke_directions(spiral)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
for spoke, label in [
    (positive_index, f"spoke {positive_index}"),
    (negative_index, f"spoke {negative_index}"),
]:
    ax.plot(
        spiral[0, :, spoke],
        spiral[1, :, spoke],
        spiral[2, :, spoke],
        ".-",
        ms=2,
        label=label,
    )

ax.scatter(
    [0],
    [0],
    [0],
    marker="*",
    s=100,
    label="k-space center",
)
ax.set_xlabel("kx")
ax.set_ylabel("ky")
ax.set_zlabel("kz")
ax.set_title("+k / -k spoke pair")
ax.legend()
plt.tight_layout()


## Short-window coverage

The plots show the first 864 consecutive spokes from each acquisition order.


In [ ]:
# Compare the angular coverage of the same number of consecutive spokes.
spiral_directions = spoke_directions(spiral)
radial_directions = spoke_directions(radial)
spiral_lon, spiral_lat = lon_lat(spiral_directions)
radial_lon, radial_lat = lon_lat(radial_directions)
n_spokes = cfg.spokes_per_frame

fig = plt.figure(figsize=(12, 4.5))

ax = fig.add_subplot(1, 2, 1, projection="mollweide")
ax.scatter(
    spiral_lon[:n_spokes],
    spiral_lat[:n_spokes],
    s=3,
)
ax.set_title(f"ZTE-STAR: first {n_spokes} spokes")
ax.grid(alpha=0.3)

ax = fig.add_subplot(1, 2, 2, projection="mollweide")
ax.scatter(
    radial_lon[:n_spokes],
    radial_lat[:n_spokes],
    s=3,
)
ax.set_title(f"Conventional radial: first {n_spokes} spokes")
ax.grid(alpha=0.3)

plt.tight_layout()


Next: run **2_reconstruction.ipynb**.
